# Chapter 3.2: Approximate Nearest Neighbor Search

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain** why exact nearest neighbor search is infeasible at scale and the need for ANN
2. **Implement** Locality-Sensitive Hashing (LSH) from scratch for cosine similarity
3. **Understand** HNSW (Hierarchical Navigable Small World) graph-based ANN
4. **Describe** IVF-PQ (Inverted File + Product Quantization) indexing
5. **Use** the Faiss library to build and query different index types
6. **Compare** speed, recall, and memory trade-offs across ANN methods
7. **Benchmark** different Faiss index types on synthetic embedding data

## Prerequisites

- Understanding of embedding spaces and similarity metrics (Chapter 3.1)
- Basic linear algebra (dot products, distances)
- Python and NumPy proficiency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part3/chapter_3.2_ann_search.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part3/chapter_3.2_ann_search.ipynb)

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Tuple, Optional

np.random.seed(42)

# We'll implement ANN from scratch first, then use Faiss
try:
    import faiss
    HAS_FAISS = True
    print(f"Faiss version: {faiss.__version__ if hasattr(faiss, '__version__') else 'available'}")
except ImportError:
    HAS_FAISS = False
    print("Faiss not installed. Will use custom implementations.")
    print("Install with: pip install faiss-cpu")

## 1. Exact vs. Approximate: Why ANN is Necessary

Given a query embedding $\mathbf{q} \in \mathbb{R}^d$ and a database of $N$ item embeddings $\{\mathbf{x}_1, \ldots, \mathbf{x}_N\}$, exact nearest neighbor search requires computing:

$$\text{NN}(\mathbf{q}) = \arg\min_{i \in [N]} \|\mathbf{q} - \mathbf{x}_i\|_2$$

This takes $O(Nd)$ time — linear in the database size. For recommendation systems with millions of items and millisecond latency requirements, this is far too slow.

| Scale | Items | Dimensions | Exact Search Time | Required Latency |
|-------|-------|-----------|-------------------|------------------|
| Small | 10K | 64 | ~1ms | <10ms |
| Medium | 1M | 128 | ~100ms | <10ms |
| Large | 100M | 256 | ~10s | <10ms |
| YouTube | 1B+ | 256 | ~100s | <10ms |

> **💡 Concept:** ANN methods trade a small amount of accuracy (recall) for orders of magnitude speedup. A typical target: 95%+ recall at 1000x speedup.

In [ ]:
# Demonstrate the scaling problem
def benchmark_exact_search(n_items_list, dim=64, n_queries=100):
    """Benchmark brute-force search at different scales."""
    times = []
    queries = np.random.randn(n_queries, dim).astype(np.float32)
    queries = queries / np.linalg.norm(queries, axis=1, keepdims=True)
    
    for n_items in n_items_list:
        database = np.random.randn(n_items, dim).astype(np.float32)
        database = database / np.linalg.norm(database, axis=1, keepdims=True)
        
        start = time.time()
        for q in queries:
            distances = np.dot(database, q)
            top_k = np.argpartition(-distances, 10)[:10]
        elapsed = time.time() - start
        times.append(elapsed / n_queries * 1000)  # ms per query
        print(f"  N={n_items:>8d}: {times[-1]:.2f} ms/query")
    
    return times

n_items_list = [1000, 5000, 10000, 50000, 100000, 500000]
print("Brute-force search benchmarks (dim=64):")
times = benchmark_exact_search(n_items_list)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_items_list, times, 'o-', color='crimson', linewidth=2, markersize=8)
ax.axhline(y=10, color='green', linestyle='--', linewidth=1.5, label='10ms latency budget')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Items', fontsize=12)
ax.set_ylabel('Latency (ms/query)', fontsize=12)
ax.set_title('Exact NN Search Latency vs. Database Size', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Locality-Sensitive Hashing (LSH)

LSH is based on a beautiful idea: design hash functions that map similar items to the same bucket with high probability.

For **cosine similarity**, we use random hyperplane hashing:

$$h_r(\mathbf{x}) = \text{sign}(\mathbf{r}^\top \mathbf{x})$$

where $\mathbf{r}$ is a random unit vector. The key property:

$$P[h_r(\mathbf{x}) = h_r(\mathbf{y})] = 1 - \frac{\theta(\mathbf{x}, \mathbf{y})}{\pi}$$

where $\theta$ is the angle between $\mathbf{x}$ and $\mathbf{y}$. Similar vectors have small angles and thus high hash collision probability.

To boost discriminative power, we use $K$ hash functions and $L$ hash tables:
- **AND-amplification**: Require all $K$ hashes to match (reduces false positives)
- **OR-amplification**: Use $L$ independent tables and union results (reduces false negatives)

> **⚠️ Common Pitfall:** LSH has high memory overhead due to multiple hash tables. For $L$ tables with $N$ items, memory is $O(LN)$. In practice, this limits LSH to moderate-scale problems.

In [ ]:
class LSHIndex:
    """Locality-Sensitive Hashing for cosine similarity.
    
    Uses random hyperplane hashing with multiple tables.
    """
    def __init__(self, dim: int, num_hashes: int = 8, num_tables: int = 10):
        self.dim = dim
        self.num_hashes = num_hashes
        self.num_tables = num_tables
        
        # Random hyperplanes for each table
        self.hyperplanes = [
            np.random.randn(num_hashes, dim).astype(np.float32)
            for _ in range(num_tables)
        ]
        
        # Hash tables: table_id -> {hash_key -> [item_indices]}
        self.tables = [defaultdict(list) for _ in range(num_tables)]
        self.data = None
    
    def _hash(self, vectors: np.ndarray, table_id: int) -> np.ndarray:
        """Compute hash for vectors using given table's hyperplanes."""
        projections = vectors @ self.hyperplanes[table_id].T  # (N, num_hashes)
        binary = (projections > 0).astype(np.int32)
        # Convert binary vector to integer hash
        powers = 2 ** np.arange(self.num_hashes)
        return (binary * powers).sum(axis=1)
    
    def build(self, data: np.ndarray):
        """Build the LSH index."""
        self.data = data.copy()
        n = data.shape[0]
        
        for t in range(self.num_tables):
            hashes = self._hash(data, t)
            for i in range(n):
                self.tables[t][hashes[i]].append(i)
    
    def query(self, query: np.ndarray, top_k: int = 10) -> Tuple[np.ndarray, np.ndarray]:
        """Query the index for nearest neighbors."""
        if query.ndim == 1:
            query = query.reshape(1, -1)
        
        # Collect candidates from all tables
        candidates = set()
        for t in range(self.num_tables):
            h = self._hash(query, t)[0]
            candidates.update(self.tables[t].get(h, []))
        
        if not candidates:
            return np.array([]), np.array([])
        
        # Re-rank candidates by exact distance
        candidate_ids = np.array(list(candidates))
        candidate_vecs = self.data[candidate_ids]
        scores = (candidate_vecs @ query.T).squeeze()
        
        if len(candidate_ids) <= top_k:
            sorted_idx = np.argsort(-scores)
        else:
            sorted_idx = np.argpartition(-scores, top_k)[:top_k]
            sorted_idx = sorted_idx[np.argsort(-scores[sorted_idx])]
        
        return candidate_ids[sorted_idx], scores[sorted_idx]


# Build and test LSH
N, D = 50000, 64
data = np.random.randn(N, D).astype(np.float32)
data = data / np.linalg.norm(data, axis=1, keepdims=True)

lsh = LSHIndex(D, num_hashes=10, num_tables=20)
start = time.time()
lsh.build(data)
print(f"LSH build time: {time.time()-start:.3f}s")

# Query
query = np.random.randn(1, D).astype(np.float32)
query = query / np.linalg.norm(query)

start = time.time()
ids, scores = lsh.query(query, top_k=10)
lsh_time = time.time() - start

# Exact search for comparison
start = time.time()
exact_scores = (data @ query.T).squeeze()
exact_top10 = np.argsort(-exact_scores)[:10]
exact_time = time.time() - start

# Compute recall
recall = len(set(ids.tolist()) & set(exact_top10.tolist())) / 10
print(f"\nLSH query time: {lsh_time*1000:.2f}ms | Exact time: {exact_time*1000:.2f}ms")
print(f"Recall@10: {recall:.2f}")
print(f"Speedup: {exact_time/lsh_time:.1f}x")

## 3. HNSW (Hierarchical Navigable Small World)

HNSW (Malkov & Yashunin, 2018) is a graph-based ANN method that builds a multi-layer proximity graph.

**Key ideas:**
1. Build a navigable small-world graph where each node connects to its nearest neighbors
2. Use a hierarchical structure: upper layers have fewer nodes for coarse navigation, lower layers have all nodes for fine-grained search
3. Search starts at the top layer and greedily descends

**Parameters:**
- $M$: Maximum number of connections per node (controls graph density)
- $ef_{\text{construction}}$: Beam width during index construction
- $ef_{\text{search}}$: Beam width during search (trade-off: higher = better recall, slower)

**Complexity:**
- Build: $O(N \log N)$
- Search: $O(\log N)$
- Memory: $O(NM)$

> **🔑 Pro Tip:** HNSW consistently achieves the best recall-speed trade-off in benchmarks. It's the default choice for most production systems when memory isn't the bottleneck.

In [ ]:
import heapq

class SimpleHNSW:
    """Simplified HNSW for educational purposes.
    
    This is a single-layer NSW (no hierarchy) to illustrate the core idea.
    Production HNSW uses multiple layers.
    """
    def __init__(self, dim: int, M: int = 16, ef_construction: int = 200):
        self.dim = dim
        self.M = M  # max connections per node
        self.ef_construction = ef_construction
        self.data = []
        self.graph = {}  # node_id -> set of neighbor ids
    
    def _distance(self, a: np.ndarray, b: np.ndarray) -> float:
        """Negative inner product (we minimize distance = maximize similarity)."""
        return -np.dot(a, b)
    
    def _search_layer(self, query: np.ndarray, entry_point: int, 
                       ef: int) -> List[Tuple[float, int]]:
        """Greedy search on the graph layer."""
        visited = {entry_point}
        candidates = [(self._distance(query, self.data[entry_point]), entry_point)]
        results = [(-candidates[0][0], entry_point)]  # max-heap (negate for max)
        
        while candidates:
            dist_c, c = heapq.heappop(candidates)
            
            # Furthest result
            if results and dist_c > -results[0][0]:
                break
            
            for neighbor in self.graph.get(c, set()):
                if neighbor not in visited:
                    visited.add(neighbor)
                    dist_n = self._distance(query, self.data[neighbor])
                    
                    if len(results) < ef or dist_n < -results[0][0]:
                        heapq.heappush(candidates, (dist_n, neighbor))
                        heapq.heappush(results, (-dist_n, neighbor))
                        if len(results) > ef:
                            heapq.heappop(results)
        
        return [(sim, idx) for sim, idx in results]
    
    def add(self, vector: np.ndarray):
        """Add a vector to the index."""
        idx = len(self.data)
        self.data.append(vector)
        self.graph[idx] = set()
        
        if idx == 0:
            return
        
        # Find nearest neighbors in current graph
        entry = 0  # simplified: always start from node 0
        neighbors = self._search_layer(vector, entry, self.ef_construction)
        
        # Connect to M nearest
        neighbors.sort(reverse=True)  # highest similarity first
        for _, n_idx in neighbors[:self.M]:
            self.graph[idx].add(n_idx)
            self.graph[n_idx].add(idx)
            
            # Prune if neighbor has too many connections
            if len(self.graph[n_idx]) > self.M * 2:
                # Keep M nearest neighbors
                dists = [(self._distance(self.data[n_idx], self.data[nn]), nn) 
                         for nn in self.graph[n_idx]]
                dists.sort()
                self.graph[n_idx] = set(nn for _, nn in dists[:self.M])
    
    def build(self, data: np.ndarray):
        """Build index from data array."""
        for i in range(len(data)):
            self.add(data[i])
    
    def query(self, query: np.ndarray, top_k: int = 10, ef_search: int = 50):
        """Query for top-k nearest neighbors."""
        results = self._search_layer(query, 0, ef_search)
        results.sort(reverse=True)
        top_results = results[:top_k]
        ids = np.array([idx for _, idx in top_results])
        scores = np.array([sim for sim, _ in top_results])
        return ids, scores

# Test on smaller dataset (our implementation is not optimized)
N_small = 5000
data_small = np.random.randn(N_small, D).astype(np.float32)
data_small = data_small / np.linalg.norm(data_small, axis=1, keepdims=True)

hnsw = SimpleHNSW(D, M=16, ef_construction=100)
start = time.time()
hnsw.build(data_small)
print(f"SimpleHNSW build time ({N_small} items): {time.time()-start:.2f}s")

# Query
ids_hnsw, scores_hnsw = hnsw.query(query.squeeze(), top_k=10, ef_search=50)

# Exact
exact_scores_small = (data_small @ query.T).squeeze()
exact_top10_small = np.argsort(-exact_scores_small)[:10]

recall_hnsw = len(set(ids_hnsw.tolist()) & set(exact_top10_small.tolist())) / 10
print(f"SimpleHNSW Recall@10: {recall_hnsw:.2f}")
print(f"Average connections per node: {np.mean([len(v) for v in hnsw.graph.values()]):.1f}")

## 4. IVF-PQ (Inverted File + Product Quantization)

IVF-PQ combines two techniques for memory-efficient ANN:

### Inverted File Index (IVF)
1. Cluster the database vectors into $C$ clusters using k-means
2. At query time, only search the $n_{\text{probe}}$ nearest clusters

### Product Quantization (PQ)
Instead of storing full $d$-dimensional vectors, PQ:
1. Split each vector into $M$ sub-vectors of dimension $d/M$
2. Quantize each sub-vector independently using $k$ centroids
3. Store only the centroid indices ($M \log_2 k$ bits per vector)

**Memory savings:**
$$\text{Full storage: } N \times d \times 4 \text{ bytes} \quad \text{PQ storage: } N \times M \text{ bytes}$$

For $d=128$, $M=8$: compression ratio = $\frac{128 \times 4}{8} = 64\times$!

> **💡 Concept:** PQ approximates the distance by decomposing it across subspaces: $\|\mathbf{x} - \mathbf{y}\|^2 \approx \sum_{m=1}^{M} \|\mathbf{x}^{(m)} - \hat{\mathbf{y}}^{(m)}\|^2$, where $\hat{\mathbf{y}}^{(m)}$ is the quantized sub-vector.

In [ ]:
class SimpleProductQuantizer:
    """Simplified Product Quantization for educational purposes."""
    
    def __init__(self, dim: int, num_subspaces: int = 8, num_centroids: int = 256):
        self.dim = dim
        self.M = num_subspaces
        self.K = num_centroids
        self.sub_dim = dim // num_subspaces
        assert dim % num_subspaces == 0
        self.centroids = None  # (M, K, sub_dim)
        self.codes = None  # (N, M) uint8
    
    def _split(self, vectors: np.ndarray) -> List[np.ndarray]:
        """Split vectors into sub-vectors."""
        return [vectors[:, i*self.sub_dim:(i+1)*self.sub_dim] for i in range(self.M)]
    
    def train(self, data: np.ndarray, n_iter: int = 20):
        """Train PQ centroids using k-means on each subspace."""
        sub_vectors = self._split(data)
        self.centroids = np.zeros((self.M, self.K, self.sub_dim), dtype=np.float32)
        
        for m in range(self.M):
            # Simple k-means
            X = sub_vectors[m]
            # Initialize centroids randomly
            indices = np.random.choice(len(X), self.K, replace=False)
            centers = X[indices].copy()
            
            for _ in range(n_iter):
                # Assign
                dists = np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)
                assignments = np.argmin(dists, axis=1)
                # Update
                for k in range(self.K):
                    mask = assignments == k
                    if mask.sum() > 0:
                        centers[k] = X[mask].mean(axis=0)
            
            self.centroids[m] = centers
    
    def encode(self, data: np.ndarray) -> np.ndarray:
        """Encode vectors to PQ codes."""
        sub_vectors = self._split(data)
        codes = np.zeros((len(data), self.M), dtype=np.uint8)
        
        for m in range(self.M):
            dists = np.sum((sub_vectors[m][:, None, :] - self.centroids[m][None, :, :]) ** 2, axis=2)
            codes[:, m] = np.argmin(dists, axis=1)
        
        self.codes = codes
        return codes
    
    def search(self, query: np.ndarray, top_k: int = 10) -> Tuple[np.ndarray, np.ndarray]:
        """Asymmetric distance computation (ADC)."""
        if query.ndim == 1:
            query = query.reshape(1, -1)
        
        sub_query = self._split(query)
        
        # Pre-compute distance lookup tables
        dist_tables = np.zeros((self.M, self.K), dtype=np.float32)
        for m in range(self.M):
            dist_tables[m] = np.sum((sub_query[m] - self.centroids[m]) ** 2, axis=1)
        
        # Compute approximate distances using lookup tables
        N = self.codes.shape[0]
        distances = np.zeros(N, dtype=np.float32)
        for m in range(self.M):
            distances += dist_tables[m, self.codes[:, m]]
        
        top_k_idx = np.argpartition(distances, top_k)[:top_k]
        top_k_idx = top_k_idx[np.argsort(distances[top_k_idx])]
        
        return top_k_idx, distances[top_k_idx]

# Demo
N_pq = 10000
data_pq = np.random.randn(N_pq, D).astype(np.float32)
data_pq = data_pq / np.linalg.norm(data_pq, axis=1, keepdims=True)

pq = SimpleProductQuantizer(D, num_subspaces=8, num_centroids=256)
print("Training PQ...")
pq.train(data_pq, n_iter=10)
codes = pq.encode(data_pq)

print(f"\nOriginal size: {data_pq.nbytes / 1024:.1f} KB")
print(f"PQ code size:  {codes.nbytes / 1024:.1f} KB")
print(f"Compression:   {data_pq.nbytes / codes.nbytes:.1f}x")

# Search
ids_pq, dists_pq = pq.search(query.squeeze(), top_k=10)
exact_scores_pq = (data_pq @ query.T).squeeze()
exact_top10_pq = np.argsort(-exact_scores_pq)[:10]

recall_pq = len(set(ids_pq.tolist()) & set(exact_top10_pq.tolist())) / 10
print(f"PQ Recall@10: {recall_pq:.2f}")

## 5. Faiss: Production-Grade ANN

Faiss (Facebook AI Similarity Search) is the industry standard for ANN. Key index types:

| Index | Description | Speed | Recall | Memory |
|-------|------------|-------|--------|--------|
| `Flat` | Brute force | Slow | 100% | High |
| `IVFFlat` | Inverted file | Medium | High | High |
| `IVFPQ` | IVF + Product Quantization | Fast | Medium | Low |
| `HNSW` | Graph-based | Fast | Very High | Medium |
| `IVF_HNSW_PQ` | IVF with HNSW quantizer + PQ | Very Fast | High | Low |

> **🔑 Pro Tip:** For most production recommendation systems, `IVF_HNSW_PQ` or `HNSW` are the go-to choices. Use `IVFPQ` when memory is constrained and `HNSW` when you need maximum recall.

In [ ]:
# Faiss benchmark (or fallback to our implementations)
N_bench = 100000
D_bench = 64
data_bench = np.random.randn(N_bench, D_bench).astype(np.float32)
data_bench = data_bench / np.linalg.norm(data_bench, axis=1, keepdims=True)

queries_bench = np.random.randn(100, D_bench).astype(np.float32)
queries_bench = queries_bench / np.linalg.norm(queries_bench, axis=1, keepdims=True)

# Ground truth
gt_scores = queries_bench @ data_bench.T
gt_top10 = np.argsort(-gt_scores, axis=1)[:, :10]

def compute_recall(predictions, ground_truth, k=10):
    """Compute average recall@k."""
    recalls = []
    for pred, gt in zip(predictions, ground_truth):
        recalls.append(len(set(pred[:k].tolist()) & set(gt[:k].tolist())) / k)
    return np.mean(recalls)

results = {}

if HAS_FAISS:
    # Index types to benchmark
    index_configs = {
        'Flat (exact)': lambda: faiss.IndexFlatIP(D_bench),
        'IVFFlat (nprobe=10)': lambda: create_ivf_flat(D_bench, 100),
        'IVFPQ (nprobe=10)': lambda: create_ivf_pq(D_bench, 100, 8),
        'HNSW32': lambda: faiss.IndexHNSWFlat(D_bench, 32),
    }
    
    def create_ivf_flat(dim, nlist):
        quantizer = faiss.IndexFlatIP(dim)
        index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        return index
    
    def create_ivf_pq(dim, nlist, m):
        quantizer = faiss.IndexFlatIP(dim)
        index = faiss.IndexIVFPQ(quantizer, dim, nlist, m, 8)
        index.metric_type = faiss.METRIC_INNER_PRODUCT
        return index
    
    for name, create_fn in index_configs.items():
        index = create_fn()
        
        # Train if needed
        if hasattr(index, 'train') and not index.is_trained:
            index.train(data_bench)
        
        # Build
        start = time.time()
        index.add(data_bench)
        build_time = time.time() - start
        
        # Set search params
        if 'IVF' in name:
            index.nprobe = 10
        
        # Query
        start = time.time()
        D_result, I_result = index.search(queries_bench, 10)
        query_time = (time.time() - start) / len(queries_bench) * 1000
        
        recall = compute_recall(I_result, gt_top10)
        results[name] = {'build': build_time, 'query_ms': query_time, 'recall': recall}
        print(f"{name:25s}: build={build_time:.3f}s  query={query_time:.3f}ms  recall@10={recall:.4f}")
else:
    # Fallback: benchmark our custom implementations
    print("Benchmarking custom implementations (install faiss-cpu for better results)")
    
    # Exact search
    start = time.time()
    exact_results = np.argsort(-(queries_bench @ data_bench.T), axis=1)[:, :10]
    exact_time = (time.time() - start) / len(queries_bench) * 1000
    results['Exact'] = {'build': 0, 'query_ms': exact_time, 'recall': 1.0}
    
    # LSH
    lsh_idx = LSHIndex(D_bench, num_hashes=12, num_tables=25)
    start = time.time()
    lsh_idx.build(data_bench)
    lsh_build = time.time() - start
    
    lsh_results = []
    start = time.time()
    for q in queries_bench:
        ids, _ = lsh_idx.query(q.reshape(1, -1), top_k=10)
        lsh_results.append(ids if len(ids) >= 10 else np.pad(ids, (0, 10 - len(ids)), constant_values=-1))
    lsh_query_time = (time.time() - start) / len(queries_bench) * 1000
    lsh_results = np.array(lsh_results)
    lsh_recall = compute_recall(lsh_results, gt_top10)
    results['LSH'] = {'build': lsh_build, 'query_ms': lsh_query_time, 'recall': lsh_recall}
    
    for name, r in results.items():
        print(f"{name:15s}: build={r['build']:.3f}s  query={r['query_ms']:.3f}ms  recall@10={r['recall']:.4f}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names = list(results.keys())
recalls = [results[n]['recall'] for n in names]
query_times = [results[n]['query_ms'] for n in names]

colors = plt.cm.Set2(np.linspace(0, 1, len(names)))

# Recall comparison
axes[0].barh(names, recalls, color=colors, edgecolor='gray')
axes[0].set_xlabel('Recall@10', fontsize=12)
axes[0].set_title('Recall@10 by Index Type', fontsize=14)
axes[0].set_xlim(0, 1.1)
for i, v in enumerate(recalls):
    axes[0].text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=10)

# Query time comparison
axes[1].barh(names, query_times, color=colors, edgecolor='gray')
axes[1].set_xlabel('Query Time (ms)', fontsize=12)
axes[1].set_title('Query Latency by Index Type', fontsize=14)
for i, v in enumerate(query_times):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=10)

plt.suptitle(f'ANN Benchmark ({N_bench:,} items, dim={D_bench})', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 6. ScaNN and Milvus Overview

### ScaNN (Google, 2020)
ScaNN (Scalable Nearest Neighbors) introduces **anisotropic vector quantization**: instead of minimizing reconstruction error uniformly, it optimizes for inner product retrieval by preserving directions that matter most for the query distribution.

Key innovation: the quantization loss is weighted by how much each direction affects the inner product ranking.

### Milvus
Milvus is an open-source vector database that provides:
- Multiple index types (IVF, HNSW, DiskANN, etc.)
- Attribute filtering alongside vector search
- Distributed architecture for billion-scale deployments
- GPU acceleration

> **🔑 Pro Tip:** When choosing an ANN system, consider: (1) scale (millions vs billions), (2) update frequency (static vs streaming), (3) filtering requirements, and (4) memory budget. Milvus or Qdrant are better for full systems; Faiss is better for embedding into your own service.

## Exercises

### 🏋️ Exercise 1: Tune LSH Parameters

Experiment with different numbers of hash functions and tables to find the best recall-speed trade-off.

In [ ]:
def lsh_parameter_sweep(data, queries, gt_top10, hash_range, table_range):
    """
    Sweep LSH parameters and record recall + query time.
    
    Args:
        data: (N, D) database vectors
        queries: (Q, D) query vectors
        gt_top10: (Q, 10) ground truth top-10 indices
        hash_range: list of num_hashes values to try
        table_range: list of num_tables values to try
    
    Returns:
        List of dicts with keys: num_hashes, num_tables, recall, query_ms
    """
    # TODO: For each (num_hashes, num_tables) combination:
    #   1. Build LSH index
    #   2. Query all queries, measure time and recall
    #   3. Store results
    # TODO: Plot recall vs query time, colored by num_tables
    pass

# hash_range = [4, 8, 12, 16]
# table_range = [5, 10, 20, 40]

### 🏋️ Exercise 2: Implement Multi-Probe LSH

Standard LSH only checks the exact hash bucket. Multi-probe LSH also checks nearby buckets (by flipping bits), achieving better recall without more tables.

In [ ]:
class MultiProbeLSH(LSHIndex):
    """LSH with multi-probe: check nearby buckets."""
    
    def query(self, query: np.ndarray, top_k: int = 10, 
              num_probes: int = 3) -> Tuple[np.ndarray, np.ndarray]:
        """
        Query with multi-probe.
        
        For each table, check the main bucket AND buckets obtained
        by flipping the bits closest to the hyperplane boundary.
        """
        # TODO: For each table:
        #   1. Compute projections (not just signs)
        #   2. Identify which bits are closest to 0 (most uncertain)
        #   3. Generate probe sequences by flipping those bits
        #   4. Check all probed buckets
        # TODO: Re-rank all candidates and return top-k
        pass

### 🏋️ Exercise 3: Benchmark Faiss Indexes with Different Parameters

Create a comprehensive benchmark comparing Faiss index types at different parameter settings.

In [ ]:
def comprehensive_faiss_benchmark(data, queries, gt_top10):
    """
    Benchmark multiple Faiss index configurations.
    
    Configs to test:
    - Flat (baseline)
    - IVFFlat with nlist=[50, 100, 500], nprobe=[1, 5, 10, 50]
    - IVFPQ with m=[4, 8, 16], nprobe=[1, 5, 10, 50]
    - HNSW with M=[8, 16, 32], efSearch=[16, 32, 64, 128]
    
    Returns:
        DataFrame-like structure with recall, latency, memory for each config
    """
    # TODO: Build each index type
    # TODO: Measure build time, query time, recall@10, and memory usage
    # TODO: Plot recall vs latency (Pareto frontier)
    # TODO: Print a summary table
    pass

# comprehensive_faiss_benchmark(data_bench, queries_bench, gt_top10)

## Summary

| Method | Time Complexity | Space | Recall | Best For |
|--------|----------------|-------|--------|----------|
| **Exact** | $O(Nd)$ | $O(Nd)$ | 100% | Small datasets (<100K) |
| **LSH** | $O(dL)$ | $O(NL)$ | 70-95% | Streaming data, simple setup |
| **HNSW** | $O(d \log N)$ | $O(NMd)$ | 95-99% | Best recall, moderate memory |
| **IVF-PQ** | $O(n_{probe} \cdot N/C)$ | $O(NM)$ | 85-95% | Memory constrained, large scale |

**Next up**: Chapter 3.3 explores Multi-Interest Retrieval — representing users with multiple embedding vectors instead of just one.